# N-CMAPSS 分阶段 RUL：基线 → 工况残差 → 图结构 → 健康瓶颈 → 迁移 → 不确定性

本 Notebook 与 `cghit_rul_model.py` 配套。每个阶段独立成 Cell，训练过程自动写入 CSV，最优模型、预测和解释图自动保存。

正式论文应分别训练并消融；不要把逐阶段训练得到的结果与从头训练的基线直接比较。

In [1]:
# Cell 1：环境修复与依赖。请先选择本项目专用 kernel，再运行。
# 如果出现 `_ARRAY_API not found`，先单独运行下一行，完成后重启 Kernel：
# %pip install --upgrade --force-reinstall "numpy==1.26.4"
# 重启后再安装其余依赖；PyTorch 请按 CUDA 版本从 pytorch.org 选择命令。
# %pip install pandas==2.2.3 h5py==3.11.0 scikit-learn==1.5.2 matplotlib==3.9.4 tqdm==4.67.1
from pathlib import Path
import sys, json, math, random, time, copy
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm.auto import tqdm

# 将 cghit_rul_model.py 放在 Notebook 同目录或当前工作目录
for p in [Path.cwd(), Path.cwd()/'outputs']:
    if (p/'cghit_rul_model.py').exists(): sys.path.insert(0, str(p)); break
from cghit_rul_model import CGHITNet, cghit_loss

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:',sys.version)
print('NumPy:',np.__version__,'| PyTorch:',torch.__version__,'| device:',DEVICE)

Python: 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
NumPy: 2.0.2 | PyTorch: 1.11.0+cu113 | device: cuda



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Program Files\Python39\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Program Files\Python39\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\Alina\AppData\Roaming\Python\Python39\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Alina\AppData\Roaming\Python\Python39\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File 

In [2]:
# Cell 2：统一实验配置
DATA_PATH=Path(r'N-CMAPSS_DS02-006.h5')
RUN_ROOT=Path('runs')/time.strftime('cghit_%Y%m%d_%H%M%S'); RUN_ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_UNITS=[2,5,10,16]
TARGET_UNIT=18
VAL_UNITS=[20]
TEST_UNITS=[11,14,15]
TARGET_FRACTION=0.20
WINDOW=50
STRIDE=10       # 通路测试=10；论文正式结果=1
BATCH_SIZE=512
NUM_WORKERS=0   # Windows + Notebook 建议先用0
EPOCHS=60
PATIENCE=12
LR=3e-4
WEIGHT_DECAY=1e-4
MC_SAMPLES=30
assert DATA_PATH.exists(), f'请修改 DATA_PATH，当前找不到 {DATA_PATH.resolve()}'
print('run dir:',RUN_ROOT.resolve())

run dir: E:\RUL\runs\cghit_20260830_184322


## Cell 3：读取和划分数据
按发动机切分；归一化器只在 source units 上拟合。目标发动机只截取前 `TARGET_FRACTION`，模拟少样本在线迁移。

In [3]:
def read_key(f,names):
    for n in names:
        if n in f: return np.asarray(f[n])
    raise KeyError(f'{names} not found; keys={list(f.keys())}')
with h5py.File(DATA_PATH,'r') as f:
    Wd=read_key(f,['W_dev']); Xd=read_key(f,['X_s_dev','Xs_dev']); Yd=read_key(f,['Y_dev']).reshape(-1).astype('float32'); Ad=read_key(f,['A_dev'])
    Wt=read_key(f,['W_test']); Xt=read_key(f,['X_s_test','Xs_test']); Yt=read_key(f,['Y_test']).reshape(-1).astype('float32'); At=read_key(f,['A_test'])
Ud=Ad[:,0].astype(int); Ut=At[:,0].astype(int)
Xdev_raw=np.concatenate([Wd,Xd],1).astype('float32'); Xtest_raw=np.concatenate([Wt,Xt],1).astype('float32')
assert Xdev_raw.shape[1]==18
scaler=StandardScaler().fit(Xdev_raw[np.isin(Ud,SOURCE_UNITS)])
Xdev=scaler.transform(Xdev_raw).astype('float32'); Xtest=scaler.transform(Xtest_raw).astype('float32')
np.savez(RUN_ROOT/'scaler.npz',mean=scaler.mean_,scale=scaler.scale_)
print(Xdev.shape,Xtest.shape,np.unique(Ud),np.unique(Ut))

(5263447, 18) (1253743, 18) [ 2  5 10 16 18 20] [11 14 15]


In [4]:
# Cell 4：惰性滑窗，避免一次复制全部窗口
class WindowDS(Dataset):
    def __init__(self,X,y,units,chosen,domain,early_fraction=None):
        self.X,self.y,self.units,self.domain=X,y,units,domain; ends=[]
        for u in chosen:
            idx=np.flatnonzero(units==u)
            if early_fraction is not None: idx=idx[:max(WINDOW,int(len(idx)*early_fraction))]
            ends.extend(idx[WINDOW-1::STRIDE])
        self.ends=np.asarray(ends,dtype=np.int64)
    def __len__(self): return len(self.ends)
    def __getitem__(self,i):
        e=int(self.ends[i]); s=e-WINDOW+1
        return (torch.from_numpy(self.X[s:e+1]),torch.tensor(self.y[e]),torch.tensor(self.domain),torch.tensor(self.units[e]))
source_ds=WindowDS(Xdev,Yd,Ud,SOURCE_UNITS,0)
target_ds=WindowDS(Xdev,Yd,Ud,[TARGET_UNIT],1,TARGET_FRACTION)
val_ds=WindowDS(Xdev,Yd,Ud,VAL_UNITS,1)
test_ds=WindowDS(Xtest,Yt,Ut,TEST_UNITS,1)
args=dict(batch_size=BATCH_SIZE,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
source_loader=DataLoader(source_ds,shuffle=True,drop_last=True,**args)
transfer_loader=DataLoader(ConcatDataset([source_ds,target_ds]),shuffle=True,drop_last=True,**args)
val_loader=DataLoader(val_ds,shuffle=False,**args); test_loader=DataLoader(test_ds,shuffle=False,**args)
print({'source':len(source_ds),'target':len(target_ds),'val':len(val_ds),'test':len(test_ds)})

{'source': 360440, 'target': 17810, 'val': 76812, 'test': 125361}


## Cell 5：基线模型
小型因果 TCN，只使用直接回归，不含工况残差、图、健康瓶颈、迁移或概率输出。

In [5]:
# Cell 5B：阶段1只增加工况残差，不加图、HI、迁移和UQ
class ConditionResidualTCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.normal_map=nn.Sequential(nn.Linear(4,32),nn.SiLU(),nn.Linear(32,14))
        self.tcn=nn.Sequential(nn.Conv1d(14,64,5,padding=2),nn.ReLU(),nn.Conv1d(64,64,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(1))
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(.2),nn.Linear(64,1))
    def forward(self,x):
        normal=self.normal_map(x[...,:4]); residual=x[...,4:]-normal
        return F.softplus(self.head(self.tcn(residual.transpose(1,2))).squeeze(-1))
condition_model=ConditionResidualTCN().to(DEVICE)
print('condition-residual params:',sum(p.numel() for p in condition_model.parameters()))

condition-residual params: 25775


In [6]:
class BaselineTCN(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Conv1d(18,64,5,padding=2),nn.ReLU(),nn.Conv1d(64,64,5,padding=2),nn.ReLU(),nn.AdaptiveAvgPool1d(1)); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(.2),nn.Linear(64,1))
    def forward(self,x): return F.softplus(self.head(self.net(x.transpose(1,2))).squeeze(-1))
baseline=BaselineTCN().to(DEVICE)
print('baseline params:',sum(p.numel() for p in baseline.parameters()))

baseline params: 26433


## Cells 6–8：结构创新

`CGHITNet` 内部按顺序执行：

1. 工况残差：学习给定飞行工况下的正常传感器值；
2. 图结构：14个传感器为节点，top-k 稀疏边描述退化传播；
3. 健康瓶颈：预测非负损伤增量，累积得到结构单调的健康指数；
4. RUL概率头：输出均值和条件方差；
5. 域分类器：通过梯度反转学习发动机间不变表征。

In [7]:
# Cell 6：完整结构实例
model=CGHITNet(n_conditions=4,n_sensors=14,node_dim=32,temporal_dim=96,top_k=4,dropout=.2,n_domains=2).to(DEVICE)
with torch.no_grad():
    demo=next(iter(source_loader))[0][:4].to(DEVICE); out=model(demo)
    print('RUL',out.mean.shape,'HI',out.health.shape,'graph',out.adjacency.shape)
    assert torch.all(out.health[:,1:]<=out.health[:,:-1]+1e-6)
print('CGHIT params:',sum(p.numel() for p in model.parameters()))

RuntimeError: Numpy is not available

In [ ]:
# Cell 7：统一评估与训练函数，逐 epoch 保存
def evaluate(net,loader,is_baseline=False):
    net.eval(); yy=[];pp=[]
    with torch.no_grad():
        for x,y,_,_ in loader:
            x=x.to(DEVICE); p=net(x) if is_baseline else net(x).mean
            yy.append(y.numpy()); pp.append(p.cpu().numpy())
    y=np.concatenate(yy); p=np.concatenate(pp)
    return mean_squared_error(y,p)**.5,mean_absolute_error(y,p)

def train_baseline(net,name='00_baseline'):
    folder=RUN_ROOT/name; folder.mkdir(exist_ok=True); opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); hist=[]; best=1e9; bad=0
    for epoch in range(1,EPOCHS+1):
        net.train(); losses=[]
        for x,y,_,_ in tqdm(source_loader,desc=f'{name} {epoch}',leave=False):
            x,y=x.to(DEVICE),y.to(DEVICE); loss=F.mse_loss(net(x),y); opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1);opt.step();losses.append(loss.item())
        rmse,mae=evaluate(net,val_loader,True); row={'epoch':epoch,'loss':np.mean(losses),'val_rmse':rmse,'val_mae':mae};hist.append(row);pd.DataFrame(hist).to_csv(folder/'history.csv',index=False);print(row)
        if rmse<best: best=rmse;bad=0;torch.save(net.state_dict(),folder/'best.pt')
        else:
            bad+=1
            if bad>=PATIENCE: break
    net.load_state_dict(torch.load(folder/'best.pt',map_location=DEVICE)); return pd.DataFrame(hist)

def train_cghit(net,use_transfer=True,name='05_full'):
    folder=RUN_ROOT/name;folder.mkdir(exist_ok=True);loader=transfer_loader if use_transfer else source_loader
    opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WEIGHT_DECAY);hist=[];best=1e9;bad=0
    healthy_threshold=float(np.quantile(Yd[np.isin(Ud,SOURCE_UNITS)],.80))
    for epoch in range(1,EPOCHS+1):
        net.train(); rows=[]
        for step,(x,y,d,_) in enumerate(tqdm(loader,desc=f'{name} {epoch}',leave=False)):
            x,y,d=x.to(DEVICE),y.to(DEVICE),d.to(DEVICE); progress=(epoch-1+step/max(1,len(loader)))/EPOCHS
            alpha=(2/(1+math.exp(-10*progress))-1) if use_transfer else 0.0
            o=net(x,alpha); loss,parts=cghit_loss(o,y,d,x[...,4:],healthy_mask=(y>=healthy_threshold),domain_weight=.1 if use_transfer else 0.0)
            opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1);opt.step();rows.append(parts)
        rmse,mae=evaluate(net,val_loader); row={'epoch':epoch,**pd.DataFrame(rows).mean().to_dict(),'val_rmse':rmse,'val_mae':mae};hist.append(row);pd.DataFrame(hist).to_csv(folder/'history.csv',index=False);print(row)
        if rmse<best:best=rmse;bad=0;torch.save(net.state_dict(),folder/'best.pt')
        else:
            bad+=1
            if bad>=PATIENCE:break
    net.load_state_dict(torch.load(folder/'best.pt',map_location=DEVICE));return pd.DataFrame(hist)

In [ ]:
# Cell 8A：训练基线。可单独运行并保存
baseline_history=train_baseline(baseline,'00_baseline')
print('baseline validation RMSE/MAE:',evaluate(baseline,val_loader,True))  # 消融阶段不查看test

In [ ]:
# Cell 8B：阶段1，仅训练“工况残差+TCN”，与基线比较 RMSE
condition_history=train_baseline(condition_model,'01_condition_residual')
print('condition-residual validation RMSE/MAE:',evaluate(condition_model,val_loader,True))  # 消融阶段不查看test

In [ ]:
# Cell 8C：仅当阶段1在5个seeds上稳定改善后，再训练“工况残差+图+健康瓶颈”
pretrain_model=CGHITNet().to(DEVICE)
structure_history=train_cghit(pretrain_model,use_transfer=False,name='01_03_structure')
print('structure validation RMSE/MAE:',evaluate(pretrain_model,val_loader))  # 仅通过预设门槛后进入下一阶段

In [ ]:
# Cell 9：迁移学习。从结构模型继续训练，加入 target 早期窗口和 DANN
model=copy.deepcopy(pretrain_model)
transfer_history=train_cghit(model,use_transfer=True,name='04_transfer')
print('transfer validation RMSE/MAE:',evaluate(model,val_loader))  # 最终方案锁定后才运行test/UQ Cell

## Cell 10：不确定性
异方差头给出 aleatoric variance；MC Dropout 给出 epistemic variance。总方差为两者之和。

In [ ]:
def mc_dropout_on(m):
    if isinstance(m,nn.Dropout):m.train()
def mc_predict(net,loader,n=30):
    net.eval();net.apply(mc_dropout_on);frames=[]
    with torch.no_grad():
        for x,y,_,u in tqdm(loader,desc='MC dropout'):
            x=x.to(DEVICE);means=[];aleas=[]
            for _ in range(n):
                o=net(x);means.append(o.mean.cpu().numpy());aleas.append(o.log_variance.exp().cpu().numpy())
            means=np.stack(means);pred=means.mean(0);var=means.var(0)+np.stack(aleas).mean(0);std=np.sqrt(var)
            frames.append(pd.DataFrame({'unit':u.numpy(),'true':y.numpy(),'pred':pred,'lower95':np.maximum(0,pred-1.96*std),'upper95':pred+1.96*std,'std':std}))
    return pd.concat(frames,ignore_index=True)
pred=mc_predict(model,test_loader,MC_SAMPLES)
metrics={'RMSE':mean_squared_error(pred.true,pred.pred)**.5,'MAE':mean_absolute_error(pred.true,pred.pred),'PICP95':((pred.true>=pred.lower95)&(pred.true<=pred.upper95)).mean(),'MPIW95':(pred.upper95-pred.lower95).mean()}
print(metrics);pred.to_csv(RUN_ROOT/'test_predictions_uq.csv',index=False);(RUN_ROOT/'metrics.json').write_text(json.dumps(metrics,indent=2))

In [ ]:
# Cell 11：黑箱解释——健康曲线、传感器重要性、传感器图
x,y,_,u=next(iter(test_loader));exp=model.explanation(x[:8].to(DEVICE))
fig,ax=plt.subplots(1,3,figsize=(16,4))
for h in exp['health_curve'].numpy():ax[0].plot(h,alpha=.75)
ax[0].set(title='Monotone health index',xlabel='time in window',ylabel='HI')
importance=exp['sensor_importance'].mean(0).numpy();ax[1].bar(np.arange(14),importance);ax[1].set(title='Mean sensor importance',xlabel='sensor index')
im=ax[2].imshow(exp['sensor_graph'].numpy(),cmap='viridis');ax[2].set(title='Learned sparse sensor graph',xlabel='source',ylabel='target');fig.colorbar(im,ax=ax[2])
plt.tight_layout();plt.savefig(RUN_ROOT/'interpretability.png',dpi=200);plt.show()
for uid,d in pred.groupby('unit'):
    d=d.iloc[::max(1,len(d)//1500)];plt.figure(figsize=(12,4));plt.plot(d.true.to_numpy(),label='true');plt.plot(d.pred.to_numpy(),label='pred');plt.fill_between(np.arange(len(d)),d.lower95,d.upper95,alpha=.25,label='95% interval');plt.title(f'unit {uid}');plt.legend();plt.grid(alpha=.2);plt.tight_layout();plt.savefig(RUN_ROOT/f'unit_{uid}.png',dpi=180);plt.show()
print('saved:',RUN_ROOT.resolve())

## Cell 12：论文消融建议

当前代码把结构模块作为一个完整机制训练。正式消融需要在 `CGHITNet` 中增加 bypass 开关，分别比较：

- Baseline；
- Baseline + condition residualizer；
- 上述 + sparse sensor graph；
- 上述 + monotone HI bottleneck；
- 上述 + target adaptation；
- 上述 + heteroscedastic NLL / MC Dropout。

每项至少运行5个随机种子，并保持 unit split、scaler、stride、early stopping 和超参数搜索预算一致。目标域适配区间必须早于目标评估区间。